# Guided (sparse) Hamiltonian Problem

Adapted from Qualtran v0.7.0, Copyright 2024 Google LLC, Apache-2.0. The tutorial uses symbolic graph oracles. The tables count declared leaves. The oracle bodies are unspecified and the tutorial does not emit quantum gates.

In [ ]:
from IPython.display import SVG, display
from sympy import symbols

from examples.algorithms.field_addition_catalog import field_signature_svg
from examples.algorithms.guided_hamiltonian_query_costs import (
    tutorial_query_graph,
    tutorial_query_upper_bound,
)
from flagquantum.algorithms.primitives.guided_hamiltonian_resource_model import GuidedHamiltonianModel
from flagquantum.core.ir.program.signatures import Signature


In [ ]:
def show_counts(counts):
    for leaf, count in counts.items():
        print(f"{leaf}: {count}")

## `GuidedHamiltonian`
Solve the guided (sparse) hamiltonian problem.

Definition 4.8 (modified with sparsity generalized to any):
In the Guided Hamiltonian problem we are given the following as input:

1. A Hamiltonian $H$ with $\|H\|_\max \le 1$, specified via a block-encoding.
2. A unitary program that takes $|0^N\rangle|0^A\rangle$ and prepares $|\Psi\rangle|0^A\rangle$.
3. Parameters $\lambda \in [-\Lambda, \Lambda]$, $\alpha \in (0, 1)$, $\gamma \in (0, 1]$.

and we should output

- YES if $\| \Pi_{\ge \lambda} (H) |\Psi\rangle \| \ge \gamma$
- NO if $\|H\| \le (1 - \alpha) \lambda$

Note that the above drops the sparse requirement, and accepts any
$(\alpha_H, \cdot, \cdot)$-block-encoding of $H$.
In the sparse Hamiltonian case, $\alpha_H = s$ (where $s$ is the sparsity).

Algorithm (Theorem 4.9):
    This uses phase estimation on the block-encoding of $e^{iHt}$, and then uses
    amplitude amplification to increase the success probability to $1 - o(1)$.

We instead directly do phase-estimation on the qubitized (Szegedy) walk operator for $H$.

#### Parameters
 - `hamiltonian`: the block-encoding of $H$
 - `guiding_state`: the unitary that prepares $|\Psi\rangle$
 - `lambd`: parameter $\lambda$
 - `alpha`: parameter $\alpha$
 - `gamma`: parameter $\gamma$

# Computing Query Costs with FlagQuantum

We will first create black-boxes for a Hamiltonian and a guiding state preparation, that can be passed to the `GuidedHamiltonian` bloq to count number of queries and gates.

## Building black-box oracles

### Graph oracles for the Hamiltonian
We first build the oracles $O_F$ and $O_H$, and use them to block-encode a sparse matrix.

In [ ]:
N, sparsity = symbols("N s", positive=True, integer=True)
entry_bits = 10
walk_ancilla = N + 1
# O_F uses two N-bit words and has a separate inverse identity.
# O_H uses two N-bit words plus a 10-bit entry word and is self-adjoint.
# Each block call declares two queries to each oracle.

position_oracle = Signature.build(i=N, k=N)
entry_oracle = Signature.build(i=N, j=N, entry=entry_bits)
print("O_F and its inverse:", position_oracle)
print("O_H:", entry_oracle)


### State-preparation Oracle for the guiding state

Point 2.  Quantum circuit that uses $G$ gates and maps $|0^N\rangle|0^A\rangle$ to $|\Psi\rangle|0^A\rangle$

In [ ]:
A, G = symbols("A G", positive=True, integer=True)
# The guide has N selection wires, A junk wires and a declared cost of G gates.

guide_signature = Signature.build(selection=N, junk=A)
print("Guide:", guide_signature)


## An example invocation
With these oracles in place, we can now invoke the `GuidedHamiltonian` algorithm.

In [ ]:
threshold, gap, overlap = symbols(r"\lambda \alpha \gamma", positive=True, real=True)
model = GuidedHamiltonianModel(
    N, walk_ancilla, N, A, sparsity, threshold, gap, overlap,
)

In [ ]:
display(SVG(field_signature_svg([("Guided Hamiltonian", model.signature, {})])))

## Query and gate costs
We will count queries to the above oracles, and arbitrary 1/2-qubit gates as described in the paper.

In [ ]:
exact = tutorial_query_graph(model, G, amplify=True).leaf_counts

In [ ]:
show_counts(exact)

### Cost from the paper
Theorem 4.9 of the paper states that the algorithm uses:
1. $Q = \widetilde{O}(s / (\gamma \alpha \lambda))$ queries to oracles for H
2. $\widetilde{O}(G/\gamma + \text{polylog}(Q)/\gamma + QN)$ gates

Let us simplify the symbolic costs obtained above and verify if they match.

In [ ]:
upper = tutorial_query_upper_bound(model, exact)
show_counts(upper)

### Cost of Phase Estimation

For simplicity, we can also look at the cost of a single phase estimation call (which is repeated $1/\gamma$ times to obtain the above algorithm).

In [ ]:
phase_exact = tutorial_query_graph(model, G, amplify=False).leaf_counts
show_counts(tutorial_query_upper_bound(model, phase_exact))